# 1 - Data Acquisition

This first part gathers the nine benchmark datasets from the 10th DIMACS Implementation Challenge as used in the articles **[1,2,3]**: 

- **Karate Club**: a classic social network with 34 nodes and 78 edges.
- **Dolphins**: a social/biological network with 62 nodes and 159 edges.
- **Political Books**: a co-purchasing network of books about US politics.
- **College Football**: a schedule-based network of college football teams.
- **Jazz**: a collaboration network of jazz musicians.
- **C. elegans**: the metabolic-reaction network of the nematode worm (453 nodes, 2025 edges).
- **E-mail**: an email interchange network from the University Rovira i Virgili.
- **PGP**: the giant component of the Pretty-Good-Privacy web of trust.
- **Condmat2003**: a condensed matter physics collaboration network.

Data source: David A. Bader, Henning Meyerhenke, Peter Sanders, Dorothea Wagner (eds.): Graph Partitioning and Graph Clustering. 10th DIMACS Implementation Challenge Workshop. February 13-14, 2012. Georgia Institute of Technology, Atlanta, GA. Contemporary Mathematics 588. American Mathematical Society and Center for Discrete Mathematics and Theoretical Computer Science, 2013.

In [2]:
using Graphs
using MatrixDepot
using SimpleWeightedGraphs
using Random

[ Info: verify download of index files...
[ Info: reading database
[ Info: adding metadata...
[ Info: adding svd data...
[ Info: writing database
[ Info: used remote sites are sparse.tamu.edu with MAT index and math.nist.gov with HTML index


In [ ]:
karate_graph = smallgraph(:karate)
dolphins_graph = SimpleGraph(matrixdepot("Newman/dolphins"))
polbooks_graph = SimpleGraph(matrixdepot("Newman/polbooks"))
football_graph = SimpleGraph(matrixdepot("Newman/football"))
jazz_graph = SimpleGraph(matrixdepot("Arenas/jazz"))

celegans_raw = matrixdepot("Arenas/celegans_metabolic")
celegans_adj = (celegans_raw .!= 0)
for i in 1:size(celegans_adj, 1)
    celegans_adj[i, i] = false
end
celegans_graph = SimpleGraph(celegans_adj)

email_graph = SimpleGraph(matrixdepot("Arenas/email"))
pgp_graph = SimpleGraph(matrixdepot("Arenas/PGPgiantcompo"))
condmat_graph = SimpleGraph(matrixdepot("Newman/cond-mat-2003"))

datasets = [
    ("Karate", karate_graph),
    ("Dolphins", dolphins_graph),
    ("Political Books", polbooks_graph),
    ("College Football", football_graph),
    ("Jazz", jazz_graph),
    ("C. elegans", celegans_graph),
    ("E-mail", email_graph),
    ("PGP", pgp_graph),
    ("Condmat2003", condmat_graph)
]

println("Datasets loaded successfully:")
for (name, g) in datasets
    println("- ", name, ": ", nv(g), " nodes, ", ne(g), " edges")
end

# 2 - Heuristic methods

## 2.1 - Simple Label Propagation (LPA)

First, I implement a basic Label Propagation Algorithm ***[article 4, section 2]***. The procedure is intentionally simple:
1. every node is initially assigned a unique label;
2. at each iteration, each node adopts the most frequent label among its neighbors;
3. when there is a tie, one of the tied labels is chosen uniformly at random.

This version does not include modularity maximization or community merging, so the result can be unstable and may yield one or two communities depending on the random seed.

In [4]:
function lpa(g; max_iter=100, seed=1)
    rng = MersenneTwister(seed)
    n = nv(g)

    current_labels = collect(1:n)

    for _ in 1:max_iter
        changed = false
        order = shuffle(rng, 1:n)

        for u in order
            nbrs = [v for v in neighbors(g, u) if v != 0]
            if isempty(nbrs)
                continue
            end

            counts = Dict{Int, Int}()
            for v in nbrs
                lab = current_labels[v]
                counts[lab] = get(counts, lab, 0) + 1
            end

            max_count = maximum(values(counts))
            candidates = [lab for (lab, count) in counts if count == max_count]
            chosen_label = current_labels[u] in candidates ? current_labels[u] : candidates[rand(rng, 1:length(candidates))]

            if current_labels[u] != chosen_label
                current_labels[u] = chosen_label
                changed = true
            end
        end

        if !changed
            break
        end
    end

    return current_labels
end


lpa (generic function with 1 method)

## 2.2 - Label Propagation with Modularity Maximization (LPAm)

I now refine the label propagation result by using the modularity objective as a local optimization criterion.
Starting from the simple LPA partition, each node is revisited and moved to the neighboring community (or a new singleton community) that yields the highest modularity gain.
When several moves give the same modularity score, one of them is selected at random. ***[article 2, section 3.2]***

### 2.2.1 - Objective function 

To compare candidate partitions for modularity maximization, we compute Newman-Girvan modularity ***[article 1, section 1, equation (2)]***.
This score measures how much more edge weight lies inside communities than would be expected by chance in a random graph with the same node strengths.

The implementation below is generic:
- unweighted graphs are treated as if every existing edge has weight `1.0`
- weighted graphs built with `SimpleWeightedGraphs.jl` use their actual edge weights
- the community assignment is given as a label vector indexed by vertex

This lets us evaluate both `Graph` and `SimpleWeightedGraph` inputs using the same objective function.  

Moreover, this algorithm iterates over labels and not vertices to speed up the execution ***[article 1, section 2.2]***.

In [ ]:
function newman_modularity(g, community)
    n = nv(g)
    @assert length(community) == n "community vector must have one label per vertex"

    has_weight = hasmethod(weight, Tuple{typeof(g), Int, Int})
    edge_weight(u, v) = has_weight ? weight(g, u, v) : 1.0

    m = 0.0
    degree_weight = zeros(Float64, n)
    for e in edges(g)
        u, v = src(e), dst(e)
        w = edge_weight(u, v)
        m += w
        degree_weight[u] += w
        degree_weight[v] += w
    end

    if m == 0.0
        return 0.0
    end

    two_m = 2.0 * m

    community_degree = Dict{Int, Float64}()
    for u in 1:n
        c = community[u]
        community_degree[c] = get(community_degree, c, 0.0) + degree_weight[u]
    end

    intra_weight = 0.0
    for e in edges(g)
        u, v = src(e), dst(e)
        if community[u] == community[v]
            intra_weight += edge_weight(u, v)
        end
    end

    Q = intra_weight / m
    for (_, Dt) in community_degree
        Q -= (Dt / two_m)^2
    end

    return Q
end

In [ ]:
# Sanity check: modularity of the trivial (one-community)
# We should obtain 0 because we didn't compute the clusters of the network yet.
function whole_graph_modularity(g)
    trivial_community = ones(Int, nv(g))
    return newman_modularity(g, trivial_community)
end

In [7]:
println("Dolphins whole-graph modularity: ", whole_graph_modularity(dolphins_graph))
println("Karate whole-graph modularity: ", whole_graph_modularity(karate_graph))

Dolphins whole-graph modularity: 0.0
Karate whole-graph modularity: 0.0


### 2.2.2 - LPAm Implementation

In [ ]:
function initialize_label_state(g, initial_labels, degrees)
    n = nv(g)
    mapping = Dict{Int, Int}()
    next_id = 1
    for lab in unique(initial_labels)
        mapping[lab] = next_id
        next_id += 1
    end
    current_labels = [mapping[lab] for lab in initial_labels]

    max_possible_labels = n
    D = zeros(Int, max_possible_labels)
    nodes_in_label = [Set{Int}() for _ in 1:max_possible_labels]

    for u in 1:n
        lab = current_labels[u]
        push!(nodes_in_label[lab], u)
        D[lab] += degrees[u]
    end

    active_labels = Set(unique(current_labels))
    return current_labels, D, nodes_in_label, active_labels
end

function neighbor_label_counts(g, u, current_labels)
    label_counts = Dict{Int, Int}()
    for v in neighbors(g, u)
        if v != 0
            lab_v = current_labels[v]
            label_counts[lab_v] = get(label_counts, lab_v, 0) + 1
        end
    end
    return label_counts
end

function ensure_label_exists!(D, nodes_in_label)
    unused_label = findfirst(==(0), D)
    if unused_label === nothing
        push!(D, 0)
        push!(nodes_in_label, Set{Int}())
        unused_label = length(D)
    end
    return unused_label
end

function choose_best_label!(u, current_lab, D, nodes_in_label, degrees, two_m, label_counts, rng)
    best_score = -Inf
    best_label = current_lab
    n_tied = 0

    for (cand, k_u_to_cand) in label_counts
        D_l = D[cand]
        score = k_u_to_cand - (degrees[u] * D_l) / two_m

        if score > best_score + 1e-12
            best_score = score
            best_label = cand
            n_tied = 1
        elseif abs(score - best_score) <= 1e-12
            n_tied += 1
            if rand(rng) < 1 / n_tied
                best_label = cand
            end
        end
    end

    unused_label = ensure_label_exists!(D, nodes_in_label)
    score_unused = 0.0 - (degrees[u] * D[unused_label]) / two_m

    if score_unused > best_score + 1e-12
        best_score = score_unused
        best_label = unused_label
        n_tied = 1
    elseif abs(score_unused - best_score) <= 1e-12
        n_tied += 1
        if rand(rng) < 1 / n_tied
            best_label = unused_label
        end
    end

    return best_label
end

function lpam_with_initial_labels(g, initial_labels; max_iter=100 * nv(g), seed=1)
    rng = MersenneTwister(seed)
    n = nv(g)
    two_m = 2.0 * ne(g)
    degrees = degree(g)

    current_labels, D, nodes_in_label, active_labels = initialize_label_state(g, initial_labels, degrees)

    # active_labels is a worklist of communities still worth re-examining; the
    # algorithm has converged once it drains to empty. max_iter is only a
    # safety cap on the number of community pops (NOT a sweep count and NOT a
    # vertex count) to guard against pathological oscillation, so it must
    # scale with the graph rather than stay a small constant -- a tiny cap
    # like 20 would silently abandon almost all communities on large graphs
    # (e.g. PGP starts with ~1900 communities after LPA).
    iter = 0
    while !isempty(active_labels) && iter < max_iter
        iter += 1

        lab = pop!(active_labels)
        if isempty(nodes_in_label[lab])
            continue
        end

        nodes_to_check = collect(nodes_in_label[lab])
        shuffle!(rng, nodes_to_check)

        for u in nodes_to_check
            if current_labels[u] != lab
                continue
            end

            current_lab = current_labels[u]
            D[current_lab] -= degrees[u]

            label_counts = neighbor_label_counts(g, u, current_labels)
            best_label = choose_best_label!(u, current_lab, D, nodes_in_label, degrees, two_m, label_counts, rng)

            D[best_label] += degrees[u]

            if best_label != current_lab
                delete!(nodes_in_label[current_lab], u)
                push!(nodes_in_label[best_label], u)
                current_labels[u] = best_label

                push!(active_labels, current_lab)
                push!(active_labels, best_label)

                for v in neighbors(g, u)
                    push!(active_labels, current_labels[v])
                end
            end
        end
    end

    return current_labels
end

function lpam(g; max_iter=100 * nv(g), seed=1)
    return lpam_with_initial_labels(g, lpa(g; seed=seed); max_iter=max_iter, seed=seed)
end


## 2.3 - Community Merging (LPAm+)


Compared with LPAm, LPAm+ adds a second refinement step in which neighboring communities are merged whenever that increases modularity ***[article 2, section 4]***. This helps avoid over-fragmentation and often yields a more stable partition than the local-move phase alone.

### 2.3.1 - LPAm+ with a classical greedy algorithm

First, I implement a classical greedy algorithm that merges exactly one pair of communities per round — whichever pair improves modularity the most — then recomputes and repeats ***[5]***. This is slow (many rounds needed) and biased: whichever community happens to look best keeps absorbing neighbors round after round, since a bigger community's degree tends to make it the most attractive partner again next round too. A better approach is described in the next section.

In [ ]:
function relabel_communities(labels)
    mapping = Dict{Int, Int}()
    next_id = 1
    relabeled = copy(labels)

    for i in eachindex(relabeled)
        lab = relabeled[i]
        if !haskey(mapping, lab)
            mapping[lab] = next_id
            next_id += 1
        end
        relabeled[i] = mapping[lab]
    end

    return relabeled
end

function communities_are_adjacent(g, labels, a, b)
    for u in 1:nv(g)
        if labels[u] != a
            continue
        end

        for v in neighbors(g, u)
            if v != 0 && labels[v] == b
                return true
            end
        end
    end

    return false
end

function merge_gain(g, labels, a, b)
    merged_labels = [lab == b ? a : lab for lab in labels]
    return newman_modularity(g, merged_labels) - newman_modularity(g, labels)
end

function best_merge_pair(g, labels)
    best_gain = 0.0
    best_pair = nothing

    communities = unique(labels)
    for i in 1:length(communities)-1
        for j in i+1:length(communities)
            a = communities[i]
            b = communities[j]

            if !communities_are_adjacent(g, labels, a, b)
                continue
            end

            gain = merge_gain(g, labels, a, b)
            if gain > best_gain + 1e-12
                best_gain = gain
                best_pair = (a, b)
            end
        end
    end

    return best_pair, best_gain
end

function merge_communities!(g, labels)
    current_labels = relabel_communities(labels)

    while true
        best_pair, best_gain = best_merge_pair(g, current_labels)
        if best_pair === nothing || best_gain <= 1e-12
            break
        end

        a, b = best_pair
        current_labels = [lab == b ? a : lab for lab in current_labels]
        current_labels = relabel_communities(current_labels)
    end

    return current_labels
end

Then, here is the main LPAm+ function:

In [ ]:
function lpam_plus(g; max_iter=100 * nv(g), seed=1)
    initial_labels = lpa(g; seed=seed)
    current_labels = lpam_with_initial_labels(g, initial_labels; max_iter=max_iter, seed=seed)

    while true
        n_communities_before = length(unique(current_labels))

        merged_labels = merge_communities!(g, current_labels)

        if length(unique(merged_labels)) == n_communities_before
            current_labels = merged_labels
            break
        end

        current_labels = lpam_with_initial_labels(g, merged_labels; max_iter=max_iter, seed=seed)
    end

    return current_labels
end

### 2.3.1 - LPAm+ with multistep greedy algorithm (MSG)

Instead of picking only the single best pair, MSG computes ΔQ for every adjacent community pair, groups equal ΔQ values into "levels," and allows merging any pair whose ΔQ falls among the top-l distinct levels (and is positive) ***[3]***.


In [11]:
function build_msg_state(g, labels)
    n = nv(g)
    relabeled = relabel_communities(labels)
    k = maximum(relabeled)

    has_weight = hasmethod(weight, Tuple{typeof(g), Int, Int})
    edge_weight(u, v) = has_weight ? weight(g, u, v) : 1.0

    L = 0.0
    degw = zeros(Float64, n)
    for e in edges(g)
        u, v = src(e), dst(e)
        w = edge_weight(u, v)
        L += w
        degw[u] += w
        degw[v] += w
    end

    d = Dict{Int, Float64}(c => 0.0 for c in 1:k)
    for u in 1:n
        d[relabeled[u]] += degw[u]
    end

    I_cut = Dict{Int, Dict{Int, Float64}}(c => Dict{Int, Float64}() for c in 1:k)
    for e in edges(g)
        u, v = src(e), dst(e)
        cu, cv = relabeled[u], relabeled[v]
        if cu != cv
            w = edge_weight(u, v)
            I_cut[cu][cv] = get(I_cut[cu], cv, 0.0) + w
            I_cut[cv][cu] = get(I_cut[cv], cu, 0.0) + w
        end
    end

    two_L2 = 2.0 * L^2
    deltaQ = Dict{Int, Dict{Int, Float64}}(c => Dict{Int, Float64}() for c in 1:k)
    for (i, row) in I_cut, (j, Iij) in row
        deltaQ[i][j] = Iij / L - (d[i] * d[j]) / two_L2   
    end

    return relabeled, d, deltaQ, L
end

function msg_merge_pair!(deltaQ, d, parent, a, b, L)
    two_L2 = 2.0 * L^2

    neighbors_ab = Set{Int}()
    for k in keys(deltaQ[a]); k != b && push!(neighbors_ab, k); end
    for k in keys(deltaQ[b]); k != a && push!(neighbors_ab, k); end

    for k in neighbors_ab
        in_a = haskey(deltaQ[a], k)
        in_b = haskey(deltaQ[b], k)
        new_val = if in_a && in_b
            deltaQ[a][k] + deltaQ[b][k]                       # i,j,k pairwise connected
        elseif in_a
            deltaQ[a][k] - (d[b] * d[k]) / two_L2             # k linked to i only
        else
            deltaQ[b][k] - (d[a] * d[k]) / two_L2             # k linked to j only
        end
        deltaQ[a][k] = new_val
        deltaQ[k][a] = new_val
        delete!(deltaQ[k], b)
    end

    delete!(deltaQ, b)
    d[a] += d[b]
    delete!(d, b)
    parent[b] = a  
end

function msg_round!(deltaQ, d, parent, L, l)
    candidates = Tuple{Int,Int,Float64}[]
    seen = Set{Tuple{Int,Int}}()
    for (i, row) in deltaQ, (j, val) in row
        if val > 1e-12
            key = i < j ? (i, j) : (j, i)
            if key ∉ seen
                push!(seen, key)
                push!(candidates, (key[1], key[2], val))
            end
        end
    end

    isempty(candidates) && return false

    sort!(candidates, by = c -> (-c[3], c[1], c[2]))

    distinct_values = unique(c[3] for c in candidates)
    threshold = distinct_values[min(l, length(distinct_values))]
    eligible = [c for c in candidates if c[3] >= threshold - 1e-12]

    touched = Set{Int}()
    merged_any = false
    for (i, j, _) in eligible
        if i in touched || j in touched   
            continue
        end
        msg_merge_pair!(deltaQ, d, parent, i, j, L)
        push!(touched, i)
        push!(touched, j)
        merged_any = true
    end

    return merged_any
end

function msg_merge_communities(g, labels; l=10)
    relabeled, d, deltaQ, L = build_msg_state(g, labels)
    parent = Dict{Int,Int}()

    while msg_round!(deltaQ, d, parent, L, l)
    end

    function find_root(x)
        while haskey(parent, x)
            x = parent[x]
        end
        return x
    end

    final_labels = [find_root(c) for c in relabeled]
    return relabel_communities(final_labels)
end

msg_merge_communities (generic function with 1 method)

In [ ]:
function lpam_plus_msg(g; max_iter=100 * nv(g), seed=1, l=10)
    initial_labels = lpa(g; seed=seed)
    current_labels = lpam_with_initial_labels(g, initial_labels; max_iter=max_iter, seed=seed)

    while true
        n_communities_before = length(unique(current_labels))

        merged_labels = msg_merge_communities(g, current_labels; l=l)

        if length(unique(merged_labels)) == n_communities_before
            current_labels = merged_labels
            break
        end

        current_labels = lpam_with_initial_labels(g, merged_labels; max_iter=max_iter, seed=seed)
    end

    return current_labels
end

# 3 - Experiments

I use the same following structure to print the results for each methods. 

In [ ]:
n_runs = 40

function print_performance_summary(method_name, method_fn, n_runs)
    println("=== $(method_name) performance over all datasets (best/avg of $(n_runs) runs) ===")
    println("Dataset               | Communities | Q_max      | Q_avg")
    println("----------------------+-------------+------------+-----------")
    for (name, g) in datasets
        best_q = -Inf
        best_communities = 0
        sum_q = 0.0
        for seed in 1:n_runs
            labels = method_fn(g, seed)
            q = newman_modularity(g, labels)
            sum_q += q
            if q > best_q
                best_q = q
                best_communities = length(unique(labels))
            end
        end
        avg_q = sum_q / n_runs
        println(rpad(name, 22), "| ", lpad(best_communities, 11), " | ", rpad(round(best_q, digits=6), 10), " | ", round(avg_q, digits=6))
    end
    println()
end


### 3.1 - LPA Performance 

In [14]:
print_performance_summary("LPA", (g, seed) -> lpa(g; seed=seed), n_runs) 

=== LPA performance over all datasets (best/avg of 40 runs) ===
Dataset               | Communities | Q_max      | Q_avg
----------------------+-------------+------------+-----------
Karate                |           4 | 0.415105   | 0.363655
Dolphins              |           5 | 0.522191   | 0.490699
Political Books       |           4 | 0.526229   | 0.502297
College Football      |          10 | 0.60457    | 0.587389
Jazz                  |           4 | 0.442395   | 0.333083
C. elegans            |          23 | 0.395484   | 0.125172
E-mail                |          57 | 0.525982   | 0.365305
PGP                   |        1913 | 0.752234   | 0.731791
Condmat2003           |        5027 | 0.625042   | 0.610228



### 3.2 - LPAm Performance 

In [15]:
print_performance_summary("LPAm", (g, seed) -> lpam(g; seed=seed), n_runs)

=== LPAm performance over all datasets (best/avg of 40 runs) ===
Dataset               | Communities | Q_max      | Q_avg
----------------------+-------------+------------+-----------
Karate                |           4 | 0.41979    | 0.381246
Dolphins              |           6 | 0.523021   | 0.500393
Political Books       |           4 | 0.526938   | 0.515698
College Football      |          10 | 0.60457    | 0.58784
Jazz                  |           4 | 0.444174   | 0.359378
C. elegans            |          23 | 0.417731   | 0.355836
E-mail                |          33 | 0.528865   | 0.456991
PGP                   |        1913 | 0.752234   | 0.731827
Condmat2003           |        5024 | 0.625059   | 0.610237



### 3.3 - LPAm+ Performance 

#### 3.2.1 - LPAm+ implemented with simple greedy algorithm

In [16]:
#print_performance_summary("LPAm+", g -> lpam_plus(g; seed=42), n_runs)

#### 3.2.2 - LPAm+ implemented with multistep greedy algorithm (MSG)

This part should find the best value of **l** for each dataset. Small datasets may required small values of l and bigger datasets, bigger values of l ... See the article ***[3]***

In [17]:
#print_performance_summary("LPAm+_MSG", (g, seed) -> lpam_plus_msg(g; seed=seed))

# 4 - References

The references below provide the main sources used throughout this notebook.

***[1]*** D. Aloise, G. Caporossi, P. Hansen, L. Liberti, S. Perron, and M. Ruiz, <i>Modularity maximization in networks by variable neighborhood search</i>, Contemporary Mathematics, vol. 588, pp. 113-127, 2013.  

***[2]*** X. Liu and T. Murata, <i>Advanced modularity-specialized label propagation algorithm for detecting communities in networks</i>, Mar. 2010.

***[3]*** P. Schuetz and A. Caflisch, <i>Efficient modularity optimization by multistep greedy algorithm and vertex mover refinement</i>, May 2008.

***[4]*** M.J. Barber and J.W. Clark, <i>Detecting network communities by propagating labels under constraints</i>, Physical Review E, vol. 80 (2009), no. 026129.

***[5]*** M. E. J. Newman, <i>Fast algorithm for detecting community structure in networks</i>, Physical Review E, vol. 69 (2004), no. 066133.